# Lesson 4 — An absent quote is not a zero one

Zero is a legal price on this exchange, so a market nobody will bid on and a market bid at nothing are different facts. Every price the engine cannot read stays null and renders as a dash with a reason.

**When it holds.** Everywhere a price is missing — which in the tails is most of the time, because nobody bids for the outcome that will not happen.

**When it fails.** A default of zero turns an unquoted tail into a one-cent market with a tight spread, and a basket summed over only its quoted legs understates the cost by exactly the legs it skipped — the direction that invents arbitrage.

| | |
|---|---|
| Lesson id | `absence` |
| Pane it appears on | `universe` (panes carry more than one lesson) |
| Code it is about | `modules/coherence/kernel/book.py`, `modules/coherence/views.py` |
| Tests that go red if it stops being true | `tests/test_coherence_lesson_0.py`, `tests/test_coherence_observe.py` |
| Pane shipped | yes |

Every cell below runs against the real kernel. Nothing here is a re-implementation:
a number this notebook prints is the number the engine would produce for the same
input. The recorded Kalshi payloads come from `tests/fixtures/coherence/`.

In [ ]:
import json
import sys
from decimal import Decimal
from pathlib import Path

# This notebook lives in notebooks/coherence_lab/ and imports the kernel two
# levels up. Found by walking upward rather than by counting parents, so the
# notebook runs from its own directory or from Part2_Infrastructure.
HERE = Path.cwd().resolve()
ROOT = next((path for path in (HERE, *HERE.parents) if (path / "modules" / "coherence" / "kernel").is_dir()), None)
if ROOT is None:
    raise SystemExit(f"no coherence kernel above {HERE}: open this notebook from inside Part2_Infrastructure")
sys.path.insert(0, str(ROOT))

FIXTURES = ROOT / "tests" / "fixtures" / "coherence"


def fixture(name: str) -> dict:
    """One recorded Kalshi response, envelope and all, exactly as it was sent.

    These are captures, not mocks. Where a number below looks odd it is because
    the exchange quoted it, and `tools/capture_kalshi_fixtures.py` re-records
    them.
    """
    return json.loads((FIXTURES / f"{name}.json").read_text(encoding="utf-8"))


print(f"kernel root       {ROOT}")
print(f"recorded fixtures {FIXTURES.is_dir()}")

## 1. Two different facts that render the same way

In [ ]:
from modules.coherence.drivers.kalshi_parse import parse_event, parse_market
from modules.coherence.kernel import coherence_index
from modules.coherence.kernel.book import Book, Level, parse_orderbook
from modules.coherence.kernel.lattice import build_component

nobody = Book(ticker="NOBODY", yes_bids=(), no_bids=())
at_zero = Book(ticker="ATZERO", yes_bids=(Level(Decimal("0.0000"), 10_000),), no_bids=())

print(f"  nobody will bid : best_yes_bid = {nobody.best_yes_bid!r}")
print(f"  bid at nothing  : best_yes_bid = {at_zero.best_yes_bid!r}   (zero is a legal price here)")
print(f"  are they equal? {nobody.best_yes_bid == at_zero.best_yes_bid}")
print()
print("  And the bug that collapses them, in one line of ordinary Python:")
print(f"    not None             -> {not None}")
print(f"    not Decimal('0.0000') -> {not Decimal('0.0000')}")
print("  `if not price:` treats a market nobody will bid on and a market bid at nothing")
print("  as the same fact. They are different facts.")

## 2. A recorded one-sided book

In [ ]:
one_sided = fixture("orderbook_one_sided")
thin_ticker = one_sided["source"].split("/markets/")[1].split("/")[0]
thin = parse_orderbook(thin_ticker, one_sided["body"]["orderbook_fp"])

print(f"  {thin_ticker}: {len(thin.yes_bids)} YES bids, {len(thin.no_bids)} NO bids")
print(f"    best YES bid {thin.best_yes_bid!r}")
print(f"    best YES ask {thin.best_yes_ask!r}")
print(f"    mid          {thin.mid!r}")
print()
print("  A real recorded book. Nobody bids for the outcome that will not happen, so in")
print("  the tails this is the ordinary case rather than a fault.")

## 3. What summing over only the quoted legs invents

In [ ]:
event = parse_event(fixture("event_mee")["body"])
component = build_component(event)
books = {market.ticker: market.top for market in event.markets}
full = sum((books[node.ticker].best_yes_ask for node in component.nodes), Decimal(0))

blind = dict(books)
dropped = component.nodes[2]
blind[dropped.ticker] = Book(ticker=dropped.ticker, yes_bids=(), no_bids=())
readable = [node for node in component.nodes if blind[node.ticker].best_yes_ask is not None]
partial = sum((blind[node.ticker].best_yes_ask for node in readable), Decimal(0))

print(f"  every leg quoted                          : {full}")
print(f"  {dropped.label} unreadable, summed over the rest : {partial}")
print(f"  the skipped leg is worth {full - partial}, and skipping it invents {Decimal(1) - partial} of arbitrage")
print()
print("  A basket summed over only its quoted legs understates the cost by exactly the")
print("  legs it skipped, which is the direction that manufactures an opportunity.")

## 4. The index withholds rather than guesses

In [ ]:
whole = coherence_index.measure(component, books)
print(f"  every leg readable : ci {whole.ci}  engine {whole.engine}")
print(f"    {whole.detail}")
print()
partial_reading = coherence_index.measure(component, blind)
print(f"  one leg unreadable : ci {partial_reading.ci!r}  engine {partial_reading.engine}")
print(f"    {partial_reading.detail}")
print()
print("  Null, with a reason. Not zero, which would read as perfect coherence and would")
print("  sit in the same column as the real measurements.")

## 5. Sixty rungs, no bids, and not one zero

In [ ]:
rungs = [parse_market(row, "KXBTCD") for row in fixture("markets_crypto")["body"]["markets"]]
bid_side = sum(1 for market in rungs if market.top.best_yes_bid is not None)
ask_side = sum(1 for market in rungs if market.top.best_yes_ask is not None)

print(f"  {len(rungs)} rungs of a recorded BTC daily ladder")
print(f"    quoted on the bid side : {bid_side}")
print(f"    quoted on the ask side : {ask_side}")
print(f"    mids available         : {sum(1 for market in rungs if market.top.mid is not None)}")
print()
print("  Every bid on this ladder is absent, so every mid is null. None of those nulls is")
print("  a probability of zero, and a recorder that wrote zeros here would produce a tape")
print("  showing an exchange that priced the whole complex at nothing.")